# Proxy de cobrança — EDA exploratória (frente Garantias)

**Frente:** D — Garantias e risco protelatório  
**Freeze:** `extracao_ref = 2026-03`  
**Público:** leitores técnicos e semi-técnicos do CTF (playground didático)

## Contexto de pesquisa

Na execução fiscal, o contribuinte pode oferecer **garantias** (carta-fiança, seguro-garantia, imóvel, depósito). A pergunta original do projeto é: *o tipo de garantia ajuda a entender risco protelatório e recuperação?*

## A lacuna

A API/dump da Inteligência Fiscal **não contém** dataset `garantia` (confirmado por meta de tabelas, HTTP 404 e ausência no OpenAPI). Sem tipo/valor/vigência de garantia, **não** treinamos nem publicamos “score de tipo de garantia”.

## O que este notebook faz

Caracteriza **proxies honestos** de cobrança — `parcelamento`, `protesto`, `ajuizamento`, `debito` (foto 2026-03) e `arrecadacao` — com:

- definição acessível de cada métrica **antes** do primeiro uso;
- leitura esperada de **cada figura antes** do gráfico;
- contagens **populacionais** (API) quando disponíveis;
- deep-dives em amostras locais `n=2000` (caminho documentado);
- figuras Plotly interativas + PNG (Kaleido) em `projects/garantias/output/figures/`.

> Playground: pergunta e protocolo ainda abertos. Não promove causalidade via garantia.


## Definições técnicas (breves) — leia antes das métricas

| Termo | Significado neste notebook |
|-------|----------------------------|
| **População (API)** | Contagem/agregado sobre **todas** as linhas do dataset via `/count`, `/aggregate` ou `/distinct`. |
| **Amostra local** | Parquet `garantias_eda_*_n2000.parquet` sob `data/extracao=2026-03/samples/`. Útil para schema, nulos e histogramas; **pode ser enviesada** pela paginação da API. |
| **Proxy de cobrança** | Variável que descreve a **trajetória** (parcelar, protestar, ajuizar, pagar), **não** o tipo de garantia. |
| **Foto de estoque** | Recorte de `debito` com `ANO_EXTRACAO=2026` e `MES_EXTRACAO=3` (~8,8 M linhas). O dump total de `debito` é histórico (~800 M). |
| **Missingness** | Fração de valores nulos numa coluna (na amostra). |
| **Ticket médio GARE** | Média de `VALOR_TOTAL_GARE` — recuperação observada, não “qualidade da garantia”. |

**Como interpretamos figuras:** títulos e eixos em português; sempre dizer se o gráfico é população ou amostra; caudas longas em valores monetários são esperadas — usamos trim no p99 só para visualização.


## 0. Setup

Carrega o cliente compartilhado (`shared.cemepi_api`), paths do projeto e diretório de figuras. O token vem do `.env` do monorepo (`CEMEPI_API_TOKEN`) — **nunca** é impresso.


In [1]:
from pathlib import Path
import sys, os, json, hashlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore", category=FutureWarning)

NB_DIR = Path.cwd()
MONO = next((p for p in [NB_DIR, *NB_DIR.parents] if (p / "shared" / "cemepi_api").exists()), NB_DIR)
sys.path.insert(0, str(MONO))
os.chdir(MONO)

from shared.cemepi_api import CemepiClient, load_settings

S = load_settings()
C = CemepiClient(S)
PROJ = MONO / "projects" / "garantias"
FIG_DIR = PROJ / "output" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
LOCAL = PROJ / ".local"
LOCAL.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR = C.sample_dir()

def save_fig(fig, stem: str, height: int = 480):
    fig.update_layout(height=height, template="plotly_white",
                      margin=dict(l=40, r=20, t=60, b=80))
    png = FIG_DIR / f"{stem}.png"
    html = LOCAL / "figures_html" / f"{stem}.html"
    html.parent.mkdir(parents=True, exist_ok=True)
    try:
        fig.write_image(str(png), scale=2)
    except Exception as e:
        print("write_image falhou:", e)
    fig.write_html(str(html), include_plotlyjs="cdn")
    fig.show()
    print("saved", png.name)

def hash_val(x, prefix="H"):
    if x is None or (isinstance(x, float) and np.isnan(x)) or str(x).strip() in ("", "nan", "None"):
        return None
    h = hashlib.sha256(f"garantias_eda:{x}".encode()).hexdigest()[:12]
    return f"[{prefix}_{h}]"

def sanitize_row(row: dict, pii=("NOME_DEVEDOR","CPF_DEVEDOR","CNPJ_DEVEDOR","CDA_COMPLETA","PEF"),
                 ids=("ID_DEBITO","ID_SOLICITACAOPARCELAMENTO","ID_GARE_SEFAZ","ID_HT_EXTRACAO_REL_SGR")):
    out = {}
    for k, v in row.items():
        if k in pii:
            pref = {"NOME_DEVEDOR":"NOME","CPF_DEVEDOR":"CPF","CNPJ_DEVEDOR":"CNPJ",
                    "CDA_COMPLETA":"CDA","PEF":"PEF"}[k]
            out[k] = hash_val(v, pref)
        elif k in ids:
            out[k] = hash_val(v, "ID")
        else:
            out[k] = None if (isinstance(v, float) and np.isnan(v)) or pd.isna(v) else v
    return out

print("extracao", f"{S.ano:04d}-{S.mes:02d}", "| base", S.base, "| sample_dir", SAMPLE_DIR)
print("token carregado:", "sim" if bool(S.token) else "NÃO")


extracao 2026-03 | base http://143.107.158.76:8010 | sample_dir /Volumes/Meedi_Etore_HD1/CEMEPI/dumps/cemepi_api/extracao=2026-03/samples
token carregado: sim


## 1. Inventário da API e ausência de `garantia`

**Antes da tabela:** listamos os datasets oficiais e testamos explicitamente `GET /v1/data/garantia`. Esperamos **não** encontrar `garantia` na lista e receber **404** no endpoint — essa é a evidência operacional da lacuna.


In [2]:
meta = C.get_json("/meta/tabelas")
names = [d["dataset"] for d in meta["datasets"]]
print("datasets:", names)
print("garantia na lista?", "garantia" in names)

r = C.session.get(f"{S.base}/v1/data/garantia", params={"limit": 1}, timeout=30)
print("GET /v1/data/garantia ->", r.status_code)
try:
    print("mensagem:", r.json().get("mensagem") or r.json())
except Exception:
    print("body:", r.text[:200])

inv = pd.DataFrame([
    {"dataset": d["dataset"], "tabela": d.get("tabela"), "linhas_api": d.get("linhas"), "chave": d.get("chave")}
    for d in meta["datasets"]
]).sort_values("linhas_api", ascending=False)
display(inv)

cnt_deb = C.get_json("/v1/data/debito/count", params={"ANO_EXTRACAO": S.ano, "MES_EXTRACAO": S.mes})
print("debito foto extracao:", cnt_deb)

gap = {
    "extracao": f"{S.ano:04d}-{S.mes:02d}",
    "datasets": names,
    "garantia": False,
    "http_garantia": r.status_code,
    "linhas": {d["dataset"]: d["linhas"] for d in meta["datasets"]},
    "debito_foto_2026_03": cnt_deb,
}
(SAMPLE_DIR / "d1_gap_garantia.json").write_text(json.dumps(gap, indent=2, ensure_ascii=False), encoding="utf-8")
print("wrote", SAMPLE_DIR / "d1_gap_garantia.json")


datasets: ['ajuizamento', 'arrecadacao', 'debito', 'faturamento', 'ingest_log', 'ingest_rejects', 'parcelamento', 'protesto', 'receita']
garantia na lista? False
GET /v1/data/garantia -> 404
mensagem: dataset desconhecido: 'garantia'


,dataset,tabela,linhas_api,chave
2,debito,DEBITO,801844918,ID_DEBITO
3,faturamento,FATURAMENTO_MASCARADO,28337827,CNPJ
1,arrecadacao,ARRECADACAO,23199305,ID_DEBITO
7,protesto,STATUS_PROTESTO,18889381,ID_DEBITO
8,receita,DEBITO_RECEITA,7754477,ID_DEBITO
6,parcelamento,SOLICITACAOPARCELAMENTO,2715674,ID_DEBITO
0,ajuizamento,AJUIZAMENTO,1850991,ID_DEBITO
5,ingest_rejects,_ingest_rejects,141837,NaN
4,ingest_log,_ingest_log,52,NaN


debito foto extracao: {'dataset': 'debito', 'total': 8821417, 'filtros': ['ANO_EXTRACAO = 2026', 'MES_EXTRACAO = 3'], 'cache': False}
wrote /Volumes/Meedi_Etore_HD1/CEMEPI/dumps/cemepi_api/extracao=2026-03/samples/d1_gap_garantia.json


### Como ler a tabela de inventário

- **linhas_api** = tamanho populacional reportado pela API (não é amostra).
- `debito` ~800 M inclui **toda a série histórica**; a foto útil é o filtro 2026-03 (~8,8 M).
- Nenhuma dessas magnitudes implica cobertura de garantia: **não há coluna de tipo/valor de garantia**.


## 2. Parcelamento — o que é esta tabela?

**Uma linha** = um **pedido** de parcelamento ligado a um `ID_DEBITO` (pode haver vários pedidos por dívida).

**Papel como proxy:** status como *Pago*, *Em andamento*, *Rompido pelo contribuinte* / *Rompido pela PGE* descreve atrito de acordo — **sem** afirmar que garantia (ou ausência dela) causou o rompimento.

### Schema (amostra) e exemplo anonimizado

A seguir: colunas, tipos, missingness na amostra `n=2000`, e **uma linha real** com IDs hasheados.


In [3]:
parc_path = SAMPLE_DIR / "garantias_eda_parcelamento_n2000.parquet"
parc = pd.read_parquet(parc_path)
print("parquet:", parc_path)
print("n =", len(parc))
schema = pd.DataFrame({
    "coluna": parc.columns,
    "dtype": [str(parc[c].dtype) for c in parc.columns],
    "null_frac": [round(float(parc[c].isna().mean()), 4) for c in parc.columns],
    "nunique": [int(parc[c].nunique(dropna=True)) for c in parc.columns],
})
display(schema)

# exemplo sem CNPJ em REGRA
cand = parc[~parc["REGRA_PARCELAMENTO"].astype(str).str.contains(r"\d{8,}", regex=True)]
ex = cand.iloc[20] if len(cand) > 20 else parc.iloc[0]
print("Exemplo de linha (anonimizado):")
display(pd.Series(sanitize_row(ex.to_dict())))


parquet: /Volumes/Meedi_Etore_HD1/CEMEPI/dumps/cemepi_api/extracao=2026-03/samples/garantias_eda_parcelamento_n2000.parquet
n = 2000


,coluna,dtype,null_frac,nunique
0,ID_SOLICITACAOPARCELAMENTO,int64,0.00,1368
1,DATA_SOLICITACAO,str,0.00,607
2,STATUS_PARCELAMENTO,str,0.00,5
3,DATA_ROMPIMENTO,str,0.52,107
4,QTD_PARCELAS,int64,0.00,71
5,TIPO_PARCELAMENTO,str,0.00,6
6,VIGENCIA_PARCELAMENTO,str,0.00,6
7,REGRA_PARCELAMENTO,str,0.00,267
8,ID_DEBITO,int64,0.00,1200
9,DT_INSCRICAO,str,0.00,1


Exemplo de linha (anonimizado):


ID_SOLICITACAOPARCELAMENTO                             [ID_21930105a2da]
DATA_SOLICITACAO                                              2021-09-17
STATUS_PARCELAMENTO                            Rompido pelo contribuinte
DATA_ROMPIMENTO                                               2022-09-28
QTD_PARCELAS                                                          43
TIPO_PARCELAMENTO                              ICMS Resolução SF/PGE- 01
VIGENCIA_PARCELAMENTO                                         ICMS_SF_01
REGRA_PARCELAMENTO            Regra Geral ICMS Resolução SF/PGE- 01 2018
ID_DEBITO                                              [ID_4b04a612983e]
DT_INSCRICAO                                                  2021-08-24
DT_CONSULTA                                                   2026-04-27
dtype: object

### Status na população (API)

**Antes do gráfico:** cada barra é a **contagem de solicitações** (população) por `STATUS_PARCELAMENTO`. Esperamos volume alto em *Pago*, *Rompido pelo contribuinte* e *Não celebrado*. Isso mapeia trajetórias de acordo — **não** tipos de garantia.


In [4]:
agg_st = C.get_json(
    "/v1/data/parcelamento/aggregate",
    params={"group_by": "STATUS_PARCELAMENTO", "metrics": "count,count_distinct:ID_DEBITO", "limit": 50},
    timeout=180,
)
df_st = pd.DataFrame(agg_st["dados"]).rename(
    columns={"STATUS_PARCELAMENTO": "status", "qtd_linhas": "qtd", "count_distinct_id_debito": "debitos_distintos"}
).sort_values("qtd", ascending=False)
display(df_st)

fig = px.bar(df_st, x="status", y="qtd", title="Parcelamento — status (população API)",
             labels={"status": "Status", "qtd": "Solicitações"}, text_auto=".2s")
fig.update_layout(xaxis_tickangle=-35, margin=dict(b=160))
save_fig(fig, "parcelamento_status_pop", height=520)


,status,qtd,debitos_distintos
0,Pago,846775,846676
1,Rompido pelo contribuinte,707031,604358
2,Não celebrado,650717,518505
3,Em andamento,384417,384377
4,Rompido pela PGE,113031,99147
5,Aguardando celebração,13698,13698
6,Rompido por falha do sistema. Aberta nova poss...,5,5


saved parcelamento_status_pop.png


### Tipo de parcelamento (população) e deep-dive amostral

**Antes dos gráficos:** (1) barras de `TIPO_PARCELAMENTO` na população mostram quais modalidades dominam; (2) histograma de `QTD_PARCELAS` na amostra mostra a dispersão do “tamanho” do acordo (mediana tipicamente alta em programas longos).


In [5]:
agg_tp = C.get_json(
    "/v1/data/parcelamento/aggregate",
    params={"group_by": "TIPO_PARCELAMENTO", "metrics": "count,count_distinct:ID_DEBITO,avg:QTD_PARCELAS", "limit": 50},
    timeout=180,
)
df_tp = pd.DataFrame(agg_tp["dados"]).rename(columns={"TIPO_PARCELAMENTO": "tipo", "qtd_linhas": "qtd"})
df_tp = df_tp.sort_values("qtd", ascending=False).head(12)
display(df_tp)

fig = px.bar(df_tp, x="tipo", y="qtd", title="Parcelamento — top tipos (população)",
             labels={"tipo": "Tipo", "qtd": "Solicitações"})
fig.update_layout(xaxis_tickangle=-40, margin=dict(b=180))
save_fig(fig, "parcelamento_tipo_top_pop", height=520)

# datas e histograma amostra
for col in ["DATA_SOLICITACAO", "DATA_ROMPIMENTO", "DT_INSCRICAO"]:
    if col in parc.columns:
        s = pd.to_datetime(parc[col], errors="coerce")
        print(f"{col}: min={s.min()} max={s.max()} null_frac={s.isna().mean():.3f}")

fig = px.histogram(parc, x="QTD_PARCELAS", nbins=40,
                   title="Parcelamento — QTD_PARCELAS (amostra n=2000)",
                   labels={"QTD_PARCELAS": "Nº de parcelas", "count": "Frequência"})
save_fig(fig, "parcelamento_qtd_parcelas_hist_sample")

# crosstab status x tipo (amostra)
ct = pd.crosstab(parc["STATUS_PARCELAMENTO"], parc["TIPO_PARCELAMENTO"])
# top 6 tipos por volume amostral
top_tipos = parc["TIPO_PARCELAMENTO"].value_counts().head(6).index
ct6 = pd.crosstab(parc["STATUS_PARCELAMENTO"], parc.loc[parc["TIPO_PARCELAMENTO"].isin(top_tipos), "TIPO_PARCELAMENTO"])
display(ct6)
fig = px.imshow(ct6, text_auto=True, aspect="auto",
                title="Amostra n=2000 — status × top tipos (contagem)",
                labels=dict(x="Tipo", y="Status", color="n"))
save_fig(fig, "parcelamento_status_x_tipo_sample", height=560)


,tipo,qtd,count_distinct_id_debito,avg_qtd_parcelas
0,ICMS Resolução SF/PGE- 01,681107,492944,41.440147
1,PPD,370269,366608,5.768414
2,IPVA Resolução 26/2023,319614,274560,6.588350
3,PEP,283543,270816,34.207238
4,PTE GRAU DE RECUPERABILIDADE,246860,237145,49.419379
5,IPVA Resolução 22/2020,219118,185066,6.779995
6,IPVA ORDINÁRIO,172094,158485,7.183057
7,PARCELAMENTO TRANSAÇÃO INDIVIDUAL,157902,137226,66.470722
8,PARCELAMENTO DE TRANSAÇÃO POR EDITAL,84201,79592,9.807152
9,PTE PEQUENOS VALORES,55239,53384,10.578740


saved parcelamento_tipo_top_pop.png
DATA_SOLICITACAO: min=2021-08-24 00:00:00 max=2026-04-20 00:00:00 null_frac=0.000


DATA_ROMPIMENTO: min=2022-01-27 00:00:00 max=2026-03-31 00:00:00 null_frac=0.520
DT_INSCRICAO: min=2021-08-24 00:00:00 max=2021-08-24 00:00:00 null_frac=0.000


saved parcelamento_qtd_parcelas_hist_sample.png


TIPO_PARCELAMENTO,ICMS Fecoep,ICMS Resolução SF/PGE- 01,PARCELAMENTO DE TRANSAÇÃO POR EDITAL,PARCELAMENTO TRANSAÇÃO INDIVIDUAL,PTE EDITAL RECUPERACAO JUDICIAL,PTE GRAU DE RECUPERABILIDADE
STATUS_PARCELAMENTO,,,,,,
Aguardando celebração,0,4,0,0,0,0
Em andamento,0,182,3,206,63,83
Não celebrado,1,411,2,56,2,24
Rompido pela PGE,0,163,0,29,0,2
Rompido pelo contribuinte,3,672,3,89,2,0


saved parcelamento_status_x_tipo_sample.png


## 3. Protesto — o que é esta tabela?

**Uma linha** = trâmite de protesto em cartório para um débito (`STATUS_PROTESTO`, `DATA_PROTESTO`, `VALOR_PROTESTADO`).

**Papel como proxy:** canal cartorário da cobrança. Agregados pesados em `protesto` podem dar **timeout (504)** — usamos `/distinct` para status populacional.

### Schema, missingness e exemplo


In [6]:
prot_path = SAMPLE_DIR / "garantias_eda_protesto_n2000.parquet"
prot = pd.read_parquet(prot_path)
print("parquet:", prot_path, "| n =", len(prot))
display(pd.DataFrame({
    "coluna": prot.columns,
    "dtype": [str(prot[c].dtype) for c in prot.columns],
    "null_frac": [round(float(prot[c].isna().mean()), 4) for c in prot.columns],
}))
cand = prot[prot["DATA_PROTESTO"].notna() & (prot["VALOR_PROTESTADO"] > 0)]
ex = cand.iloc[10] if len(cand) > 10 else prot.iloc[0]
print("Exemplo (anonimizado):")
display(pd.Series(sanitize_row(ex.to_dict())))


parquet: /Volumes/Meedi_Etore_HD1/CEMEPI/dumps/cemepi_api/extracao=2026-03/samples/garantias_eda_protesto_n2000.parquet | n = 2000


,coluna,dtype,null_frac
0,ID_DEBITO,int64,0.0000
1,DT_INSCRICAO,str,0.0000
2,STATUS_PROTESTO,str,0.0000
3,DATA_PROTESTO,str,0.1215
4,VALOR_PROTESTADO,float64,0.0000


Exemplo (anonimizado):


ID_DEBITO                                           [ID_f549fc63669f]
DT_INSCRICAO                                               2017-07-15
STATUS_PROTESTO     Aguardando pagamento dos emolumentos e cancela...
DATA_PROTESTO                                              2020-07-16
VALOR_PROTESTADO                                               446.08
dtype: object

### Status na população e distribuição de valor (amostra)

**Antes dos gráficos:**  
1) Barras dos **top status** via `/distinct` — população. Status como “Cartório protestou”, “Pago Cartório”, “Protesto Cancelado” descrevem o funil cartorário.  
2) Histograma de `VALOR_PROTESTADO` (amostra, trim p99) — cauda longa esperada; mediana tipicamente na casa de centenas/milhares de reais.


In [7]:
dist = C.get_json("/v1/data/protesto/distinct", params={"column": "STATUS_PROTESTO", "limit": 50}, timeout=180)
df_pr = pd.DataFrame(dist["valores"]).rename(columns={"valor": "status", "qtd": "qtd"}).sort_values("qtd", ascending=False)
display(df_pr.head(15))

fig = px.bar(df_pr.head(10), x="status", y="qtd", title="Protesto — top 10 status (população /distinct)",
             labels={"status": "Status", "qtd": "Registros"})
fig.update_layout(xaxis_tickangle=-35, margin=dict(b=180))
save_fig(fig, "protesto_status_top10_pop", height=560)

s = pd.to_datetime(prot["DATA_PROTESTO"], errors="coerce")
print(f"DATA_PROTESTO amostra: min={s.min()} max={s.max()} null_frac={s.isna().mean():.4f}")
print(f"DT_INSCRICAO amostra: {pd.to_datetime(prot['DT_INSCRICAO'], errors='coerce').min()} → {pd.to_datetime(prot['DT_INSCRICAO'], errors='coerce').max()}")

p99 = prot["VALOR_PROTESTADO"].quantile(0.99)
fig = px.histogram(prot[prot["VALOR_PROTESTADO"] <= p99], x="VALOR_PROTESTADO", nbins=50,
                   title=f"Protesto — VALOR_PROTESTADO (amostra n=2000, trim ≤ p99={p99:,.0f})",
                   labels={"VALOR_PROTESTADO": "R$ protestado"})
save_fig(fig, "protesto_valor_hist_sample")
print(prot["VALOR_PROTESTADO"].describe(percentiles=[0.5, 0.95, 0.99]).round(2))


,status,qtd
0,Aguardando pagamento dos emolumentos e cancela...,6485125
1,Cartório protestou,5674265
2,Protesto Cancelado,2270004
3,Pago Cartório,1504413
4,Cartório nao protestou,1374504
5,Aguardando geração do arquivo,708353
6,Débito protestado quitado via IEPTB-SP,283067
7,Aguardando seleção para próxima remessa de pro...,270509
8,Solicitado Cancelamento,215931
9,Enviado para cartório de protesto,51821


saved protesto_status_top10_pop.png


DATA_PROTESTO amostra: min=2016-02-10 00:00:00 max=2026-03-12 00:00:00 null_frac=0.1215
DT_INSCRICAO amostra: 2016-01-05 00:00:00 → 2026-02-27 00:00:00


saved protesto_valor_hist_sample.png
count       2000.00
mean       11143.70
std       216438.07
min            0.00
50%          910.92
95%        12501.20
99%       132466.71
max      9452723.37
Name: VALOR_PROTESTADO, dtype: float64


## 4. Ajuizamento — o que é esta tabela?

**Uma linha** = um débito que virou **processo** (PEF), com data e comarca.

**Papel como proxy:** entrada na via judicial. Complementa o status de ajuizamento visto no estoque `debito`.


In [8]:
aju_path = SAMPLE_DIR / "garantias_eda_ajuizamento_n2000.parquet"
aju = pd.read_parquet(aju_path)
print("parquet:", aju_path, "| n =", len(aju))
display(pd.DataFrame({
    "coluna": aju.columns,
    "dtype": [str(aju[c].dtype) for c in aju.columns],
    "null_frac": [round(float(aju[c].isna().mean()), 4) for c in aju.columns],
}))
ex = aju.iloc[12]
print("Exemplo (anonimizado):")
display(pd.Series(sanitize_row(ex.to_dict())))

s = pd.to_datetime(aju["DT_AJUIZAMENTO"], errors="coerce")
print(f"DT_AJUIZAMENTO: {s.min()} → {s.max()} | null_frac={s.isna().mean():.4f}")

# cobertura temporal amostral
tmp = aju.assign(ano=s.dt.year)
vc = tmp["ano"].value_counts().sort_index().reset_index()
vc.columns = ["ano", "qtd"]
fig = px.bar(vc, x="ano", y="qtd", title="Ajuizamento — ano de DT_AJUIZAMENTO (amostra n=2000)",
             labels={"ano": "Ano", "qtd": "Registros"})
save_fig(fig, "ajuizamento_ano_sample")


parquet: /Volumes/Meedi_Etore_HD1/CEMEPI/dumps/cemepi_api/extracao=2026-03/samples/garantias_eda_ajuizamento_n2000.parquet | n = 2000


,coluna,dtype,null_frac
0,ID_DEBITO,int64,0.0
1,DT_INSCRICAO,str,0.0
2,PEF,str,0.0
3,DT_AJUIZAMENTO,str,0.0
4,NOMECOMARCA,str,0.0


Exemplo (anonimizado):


ID_DEBITO              [ID_adc1fce03d77]
DT_INSCRICAO                  2016-01-04
PEF                   [PEF_50d1c1c3e77a]
DT_AJUIZAMENTO                2016-09-28
NOMECOMARCA       Comarca de Nova Odessa
dtype: str

DT_AJUIZAMENTO: 2016-01-13 00:00:00 → 2024-01-18 00:00:00 | null_frac=0.0000


saved ajuizamento_ano_sample.png


### Comarcas (população)

**Antes do gráfico:** top comarcas por volume populacional. Concentração em foros de São Paulo/Capital é esperada; encoding quebrado (“SÃ£o”) pode aparecer em alguns rótulos da API — anotamos como qualidade de dado, sem “corrigir” inventando.


In [9]:
agg_aj = C.get_json(
    "/v1/data/ajuizamento/aggregate",
    params={"group_by": "NOMECOMARCA", "metrics": "count,count_distinct:ID_DEBITO", "limit": 20},
    timeout=180,
)
df_aj = pd.DataFrame(agg_aj["dados"]).rename(columns={"NOMECOMARCA": "comarca", "qtd_linhas": "qtd"}).sort_values("qtd", ascending=False)
display(df_aj)

fig = px.bar(df_aj.head(12), x="comarca", y="qtd", title="Ajuizamento — top comarcas (população)",
             labels={"comarca": "Comarca", "qtd": "Débitos"})
fig.update_layout(xaxis_tickangle=-35, margin=dict(b=200))
save_fig(fig, "ajuizamento_comarca_top_pop", height=560)


,comarca,qtd,count_distinct_id_debito
0,Comarca de São Paulo - Foro das Execuções Fisc...,528272,528272
1,Vara das Execuções Fiscais Estaduais da Comarc...,412475,412475
2,Comarca de Guarulhos,57290,57290
3,Comarca de Barueri,48037,48037
4,Comarca de SÃ£o Paulo - Foro das ExecuÃ§Ãµes F...,47637,47637
5,Comarca de Poá,39812,39812
6,Comarca de Campinas,36071,36071
7,Comarca de Osasco,33624,33624
8,Comarca de São Bernardo do Campo,27238,27238
9,Comarca de Diadema,23974,23974


saved ajuizamento_comarca_top_pop.png


## 5. Débito (foto 2026-03) — o que é esta tabela?

**Uma linha** = uma dívida **inscrita na foto mensal** (`ANO_EXTRACAO`/`MES_EXTRACAO`). Contém tipo, situação, status de ajuizamento e valores. Há PII (`NOME_DEVEDOR`, CPF/CNPJ) — **só usamos hash** em exemplos.

**Papel como proxy:** estoque e composição do que pode ser cobrado / ajuizado.


In [10]:
deb_path = SAMPLE_DIR / "garantias_eda_debito_n2000.parquet"
deb = pd.read_parquet(deb_path)
print("parquet:", deb_path, "| n =", len(deb))
# não exibir colunas PII em schema display com valores — só nomes
display(pd.DataFrame({
    "coluna": deb.columns,
    "dtype": [str(deb[c].dtype) for c in deb.columns],
    "null_frac": [round(float(deb[c].isna().mean()), 4) for c in deb.columns],
}))
ex = deb.iloc[3]
print("Exemplo (anonimizado):")
display(pd.Series(sanitize_row(ex.to_dict())))


parquet: /Volumes/Meedi_Etore_HD1/CEMEPI/dumps/cemepi_api/extracao=2026-03/samples/garantias_eda_debito_n2000.parquet | n = 2000


,coluna,dtype,null_frac
0,ID_HT_EXTRACAO_REL_SGR,int64,0.000
1,ANO_EXTRACAO,int64,0.000
2,MES_EXTRACAO,int64,0.000
3,ID_DEBITO,int64,0.000
4,CDA_COMPLETA,str,0.000
5,DATA_INSCRICAO,str,0.000
6,SITUACAO_DEBITO,str,0.000
7,TIPO_DEBITO,str,0.000
8,STATUS_AJUIZAMENTO_DEBITO,str,0.000
9,VALOR_COM_VH,str,0.000


Exemplo (anonimizado):


ID_HT_EXTRACAO_REL_SGR                [ID_15f7a621b2f9]
ANO_EXTRACAO                                       2026
MES_EXTRACAO                                          3
ID_DEBITO                             [ID_d0435609a6ea]
CDA_COMPLETA                         [CDA_02ae909434f4]
DATA_INSCRICAO                    2021-08-18 00:00:00.0
SITUACAO_DEBITO                                Inscrito
TIPO_DEBITO                                        IPVA
STATUS_AJUIZAMENTO_DEBITO     Liberado para ajuizamento
VALOR_COM_VH                                     752.73
VALOR_SEM_HONORARIOS                              684.3
VALOR_VH                                            0.0
HONORARIOS_ADMINISTRATIVOS                        68.43
NOME_DEVEDOR                        [NOME_f7dc70af41ca]
CNPJ_DEVEDOR                                       None
CPF_DEVEDOR                          [CPF_faf7a3778f93]
dtype: object

### Tipo e status de ajuizamento na população (foto)

**Antes dos gráficos:**  
1) Top `TIPO_DEBITO` na foto — IPVA e ICMS Declarado costumam dominar em contagem.  
2) Pizza/barras de `STATUS_AJUIZAMENTO_DEBITO` — “Liberado para ajuizamento” vs “Ajuizado” descreve o estoque judicializável, **não** garantia.


In [11]:
agg_tipo = C.get_json(
    "/v1/data/debito/aggregate",
    params={"group_by": "TIPO_DEBITO", "metrics": "count,sum:VALOR_SEM_HONORARIOS",
            "ANO_EXTRACAO": S.ano, "MES_EXTRACAO": S.mes, "limit": 30},
    timeout=180,
)
df_tipo = pd.DataFrame(agg_tipo["dados"]).rename(columns={"TIPO_DEBITO": "tipo", "qtd_linhas": "qtd"}).sort_values("qtd", ascending=False)
display(df_tipo.head(15))
fig = px.bar(df_tipo.head(12), x="tipo", y="qtd", title="Débito foto 2026-03 — top tipos (população)",
             labels={"tipo": "Tipo", "qtd": "Débitos"})
fig.update_layout(xaxis_tickangle=-35, margin=dict(b=160))
save_fig(fig, "debito_tipo_top_extracao_2026_03", height=520)

agg_saj = C.get_json(
    "/v1/data/debito/aggregate",
    params={"group_by": "STATUS_AJUIZAMENTO_DEBITO", "metrics": "count",
            "ANO_EXTRACAO": S.ano, "MES_EXTRACAO": S.mes, "limit": 20},
    timeout=180,
)
df_saj = pd.DataFrame(agg_saj["dados"]).rename(columns={"STATUS_AJUIZAMENTO_DEBITO": "status", "qtd_linhas": "qtd"})
display(df_saj)
fig = px.pie(df_saj, names="status", values="qtd", title="Débito foto 2026-03 — status ajuizamento (população)")
save_fig(fig, "debito_status_ajuizamento_2026_03")

agg_sit = C.get_json(
    "/v1/data/debito/aggregate",
    params={"group_by": "SITUACAO_DEBITO", "metrics": "count",
            "ANO_EXTRACAO": S.ano, "MES_EXTRACAO": S.mes, "limit": 20},
    timeout=180,
)
df_sit = pd.DataFrame(agg_sit["dados"]).rename(columns={"SITUACAO_DEBITO": "situacao", "qtd_linhas": "qtd"}).sort_values("qtd", ascending=False)
display(df_sit)

# amostra: distribuição de valor (trim)
v = pd.to_numeric(deb["VALOR_SEM_HONORARIOS"], errors="coerce")
p99 = v.quantile(0.99)
fig = px.histogram(v[v <= p99], nbins=50,
                   title=f"Débito — VALOR_SEM_HONORARIOS (amostra n=2000, trim ≤ p99)",
                   labels={"value": "R$ sem honorários"})
save_fig(fig, "debito_valor_hist_sample")

# crosstab amostra tipo x status aj
top_t = deb["TIPO_DEBITO"].value_counts().head(6).index
ct = pd.crosstab(deb.loc[deb["TIPO_DEBITO"].isin(top_t), "TIPO_DEBITO"], deb.loc[deb["TIPO_DEBITO"].isin(top_t), "STATUS_AJUIZAMENTO_DEBITO"])
display(ct)
fig = px.imshow(ct, text_auto=True, aspect="auto",
                title="Amostra n=2000 — tipo × status ajuizamento",
                labels=dict(color="n"))
save_fig(fig, "debito_tipo_x_status_aj_sample", height=480)


,tipo,qtd,sum_valor_sem_honorarios
0,IPVA,5702184,7.708046e+09
1,ICMS Declarado,2284655,1.194941e+11
2,Taxa Judiciária,616276,1.300103e+09
3,ICMS Autuação,59964,3.290693e+11
4,Multa Penal,56610,2.051234e+09
5,Multas,50736,7.243323e+09
6,ICMS Fecoep,7920,5.358654e+07
7,Multa de Nota Fiscal Paulista,7828,2.123634e+08
8,Multa Ipca,6844,3.748123e+09
9,Multa CBPMESP,6755,6.244217e+07


saved debito_tipo_top_extracao_2026_03.png


,status,qtd
0,Liberado para ajuizamento,6680823
1,Ajuizado,2140588
2,Aguardando liberação para ajuizar,6


saved debito_status_ajuizamento_2026_03.png


,situacao,qtd
0,Inscrito,8759186
1,Suspenso,47721
2,Processo sobrestado - artigo 40 (por decisao j...,12655
3,Selecionado para Saneamento,1439
4,Suspenso outros motivos ( por decisao judicial...,416


saved debito_valor_hist_sample.png


STATUS_AJUIZAMENTO_DEBITO,Ajuizado,Liberado para ajuizamento
TIPO_DEBITO,,
ICMS Autuação,8,2
ICMS Declarado,300,169
IPVA,104,1251
Multa Penal,3,10
Multas,4,4
Taxa Judiciária,1,129


saved debito_tipo_x_status_aj_sample.png


## 6. Arrecadação (GARE) — o que é esta tabela?

**Uma linha** = um **pagamento** (`VALOR_TOTAL_GARE` e componentes) ligado a um débito.

**Papel como proxy / outcome:** recuperação observada. Ticket médio maior em débitos já ajuizados **não** significa “melhor garantia” — é associação descritiva com estágio da cobrança.


In [12]:
arr_path = SAMPLE_DIR / "garantias_eda_arrecadacao_n2000.parquet"
arr = pd.read_parquet(arr_path)
print("parquet:", arr_path, "| n =", len(arr))
display(pd.DataFrame({
    "coluna": arr.columns,
    "dtype": [str(arr[c].dtype) for c in arr.columns],
    "null_frac": [round(float(arr[c].isna().mean()), 4) for c in arr.columns],
}))
cand = arr[arr["VALOR_TOTAL_GARE"] > 100].sort_values("VALOR_TOTAL_GARE")
ex = cand.iloc[len(cand)//2] if len(cand) else arr.iloc[0]
print("Exemplo (anonimizado):")
display(pd.Series(sanitize_row(ex.to_dict())))


parquet: /Volumes/Meedi_Etore_HD1/CEMEPI/dumps/cemepi_api/extracao=2026-03/samples/garantias_eda_arrecadacao_n2000.parquet | n = 2000


,coluna,dtype,null_frac
0,ANO,int64,0.0
1,MES,int64,0.0
2,ID_DEBITO,int64,0.0
3,CDA_COMPLETA,str,0.0
4,STATUS_AJUIZAMENTO_DEBITO,str,0.0
5,ID_GARE_SEFAZ,int64,0.0
6,DATA_ARRECADACAO_GARE,str,0.0
7,VALOR_TOTAL_GARE,float64,0.0
8,VALOR_RECEITA,float64,0.0
9,JUROS_MORA,float64,0.0


Exemplo (anonimizado):


ANO                                          2016
MES                                            12
ID_DEBITO                       [ID_9e01a2ad129b]
CDA_COMPLETA                   [CDA_544b6e230fcc]
STATUS_AJUIZAMENTO_DEBITO                Ajuizado
ID_GARE_SEFAZ                   [ID_ea7e39dd1424]
DATA_ARRECADACAO_GARE         2016-12-26T00:00:00
VALOR_TOTAL_GARE                           457.68
VALOR_RECEITA                              186.85
JUROS_MORA                                  88.17
MULTA_MORA                                  18.68
VALOR_ACRESCIMO_FINANCEIRO                 143.46
HONORARIOS_ADMINISTRATIVOS                   None
HONORARIOS_ADVOCATICIOS                 20.521479
dtype: object

### Ticket médio por status de ajuizamento (população)

**Antes do gráfico:** barras de `avg(VALOR_TOTAL_GARE)` por `STATUS_AJUIZAMENTO_DEBITO`. Interpretação correta: **recuperação condicionada ao estágio**, não score de garantia. A amostra local pode ter janela de datas estreita (viés de paginação) — por isso o agregado populacional é a referência.


In [13]:
agg_ar = C.get_json(
    "/v1/data/arrecadacao/aggregate",
    params={"group_by": "STATUS_AJUIZAMENTO_DEBITO",
            "metrics": "count,sum:VALOR_TOTAL_GARE,avg:VALOR_TOTAL_GARE", "limit": 20},
    timeout=180,
)
df_ar = pd.DataFrame(agg_ar["dados"]).rename(columns={
    "STATUS_AJUIZAMENTO_DEBITO": "status",
    "qtd_linhas": "qtd",
    "sum_valor_total_gare": "soma_gare",
    "avg_valor_total_gare": "ticket_medio",
})
display(df_ar)

fig = px.bar(df_ar, x="status", y="ticket_medio",
             title="Arrecadação — ticket médio GARE por status ajuizamento (população)",
             labels={"status": "Status ajuizamento", "ticket_medio": "Ticket médio (R$)"},
             text=df_ar["ticket_medio"].round(0))
fig.update_layout(xaxis_tickangle=-20, margin=dict(b=120))
save_fig(fig, "arrecadacao_ticket_por_status_aj_pop")

fig = px.bar(df_ar, x="status", y="qtd",
             title="Arrecadação — volume de GARE por status ajuizamento (população)",
             labels={"status": "Status", "qtd": "Pagamentos"})
save_fig(fig, "arrecadacao_volume_por_status_aj_pop")

s = pd.to_datetime(arr["DATA_ARRECADACAO_GARE"], errors="coerce")
print(f"DATA_ARRECADACAO_GARE amostra: {s.min()} → {s.max()} (janela pode ser estreita — viés de sample)")
v = arr["VALOR_TOTAL_GARE"]
p99 = v.quantile(0.99)
fig = px.histogram(arr[arr["VALOR_TOTAL_GARE"] <= p99], x="VALOR_TOTAL_GARE", nbins=50,
                   title="Arrecadação — VALOR_TOTAL_GARE (amostra n=2000, trim p99)",
                   labels={"VALOR_TOTAL_GARE": "R$ GARE"})
save_fig(fig, "arrecadacao_valor_hist_sample")


,status,qtd,soma_gare,ticket_medio
0,Liberado para ajuizamento,16882882,2.174052e+10,1287.725670
1,Ajuizado,6316297,2.281144e+10,3611.520458
2,Aguardando liberação para ajuizar,126,1.313015e+05,1042.075322


saved arrecadacao_ticket_por_status_aj_pop.png


saved arrecadacao_volume_por_status_aj_pop.png
DATA_ARRECADACAO_GARE amostra: 2016-12-01 00:00:00 → 2016-12-29 00:00:00 (janela pode ser estreita — viés de sample)


saved arrecadacao_valor_hist_sample.png


## 7. O que os proxies podem e não podem dizer

| Podem (com honestidade) | Não podem |
|-------------------------|-----------|
| Descrever funis de parcelamento, protesto e ajuizamento | Substituir **tipo** de garantia |
| Medir volumes e tickets de recuperação (GARE) | Inferir causalidade garantia → protelação |
| Preparar features de trajetória para um futuro modelo | Publicar “score de garantia” agora |
| Documentar a lacuna com evidência reproduzível | Inventar colunas que não existem na API |

### Viabilidade

**go-with-gaps:** seguimos com EDA e preparação analítica **sem** fingir cobertura do constructo central.

### Persistência dos agregados

A célula seguinte grava um snapshot em `projects/garantias/.local/` para o briefing (sem token, sem PII).


In [14]:
snapshot = {
    "extracao_ref": f"{S.ano:04d}-{S.mes:02d}",
    "parcelamento_by_status": agg_st,
    "parcelamento_by_tipo_top": df_tp.to_dict(orient="records"),
    "protesto_status_top": df_pr.head(20).to_dict(orient="records"),
    "ajuizamento_comarca_top": df_aj.to_dict(orient="records"),
    "debito_by_tipo_top": df_tipo.head(20).to_dict(orient="records"),
    "debito_by_status_aj": df_saj.to_dict(orient="records"),
    "debito_by_situacao": df_sit.to_dict(orient="records"),
    "arrecadacao_by_status_aj": df_ar.to_dict(orient="records"),
    "sample_paths": {
        "parcelamento": str(parc_path),
        "protesto": str(prot_path),
        "ajuizamento": str(aju_path),
        "debito": str(deb_path),
        "arrecadacao": str(arr_path),
    },
    "figures_dir": str(FIG_DIR),
}
out = LOCAL / "eda_population_aggs_refresh.json"
out.write_text(json.dumps(snapshot, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("wrote", out)

sample_stats = {
    "parcelamento": {"n": len(parc), "null_frac": parc.isna().mean().round(4).to_dict()},
    "protesto": {"n": len(prot), "null_frac": prot.isna().mean().round(4).to_dict()},
    "ajuizamento": {"n": len(aju), "null_frac": aju.isna().mean().round(4).to_dict()},
    "debito": {"n": len(deb), "null_frac": deb.isna().mean().round(4).to_dict()},
    "arrecadacao": {"n": len(arr), "null_frac": arr.isna().mean().round(4).to_dict()},
}
(LOCAL / "eda_sample_stats_refresh.json").write_text(json.dumps(sample_stats, indent=2, ensure_ascii=False), encoding="utf-8")
print("figures:", sorted(p.name for p in FIG_DIR.glob("*.png")))


wrote /Users/etorebraga/Code/cemepi-ctf-intel-fiscal/projects/garantias/.local/eda_population_aggs_refresh.json


figures: ['ajuizamento_ano_sample.png', 'ajuizamento_comarca_top_pop.png', 'arrecadacao_ticket_por_status_aj_pop.png', 'arrecadacao_valor_hist_sample.png', 'arrecadacao_volume_por_status_aj_pop.png', 'debito_status_ajuizamento_2026_03.png', 'debito_tipo_top_extracao_2026_03.png', 'debito_tipo_x_status_aj_sample.png', 'debito_valor_hist_sample.png', 'parcelamento_qtd_parcelas_hist_sample.png', 'parcelamento_status_pop.png', 'parcelamento_status_x_tipo_sample.png', 'parcelamento_tipo_top_pop.png', 'protesto_status_top10_pop.png', 'protesto_valor_hist_sample.png']


## 8. Próximos passos analíticos (não são “pedidos de dados”)

1. Cruzar, no nível `ID_DEBITO`, trajetórias amostrais: parcelamento rompido × presença em protesto × ajuizamento × GARE (com n e viés documentados).  
2. Separar análises por `TIPO_DEBITO` (IPVA vs ICMS) — heterogeneidade de funil.  
3. Estudar tempos (solicitação→rompimento; inscrição→ajuizamento; inscrição→pagamento) na medida em que as amostras/API permitirem.  
4. Manter qualquer modelagem futura rotulada como **proxy de cobrança**, nunca como score de tipo de garantia.  
5. Se/quando `garantia` existir no dump: nova rodada de caracterização por tipo e só então hipóteses de score.

---

*Conteúdo, experimentos e conclusões: Étore. Formatação e organização didática do notebook: assistência de IA.*
